# multiply-back composite — cx29: multiply_back as elementwise chain rule — dL/dx = grad_out * y

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `multiply-back`, `chain-rule-elementwise`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "multiply-back"
DD_ATOM_IDS = ["multiply-back", "chain-rule-elementwise"]
DD_SUBTOPICS = ["Backprop: multiply_back", "Backprop: Elementwise chain rule"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The elementwise chain rule says: for `out = f(x, y)`, `dL/dx = grad_out * (∂out/∂x)`, all elementwise. For `f = multiply`, `∂out/∂x = y` and `∂out/∂y = x`, giving the multiply-back formulas `dL/dx = grad_out * y` and `dL/dy = grad_out * x`. So `multiply_back` is the simplest concrete instance of the elementwise chain rule applied to a binary op — no special derivative, the OTHER operand IS the derivative.

Composing them: implement both `multiply_back0` and `multiply_back1` and verify against the elementwise chain-rule pattern (grad_out * local-derivative) and against torch's autograd.

### Composite Exercise — multiply_back as elementwise chain rule — dL/dx = grad_out * y

**Atoms exercised together**: `multiply-back`, `chain-rule-elementwise`

Implement `cx29_multiply_back(grad_out, out, x, y, argnum)` that:

- if `argnum == 0`: returns `grad_out * y` (chain rule with local derivative `y`)
- if `argnum == 1`: returns `grad_out * x` (chain rule with local derivative `x`)

Same shapes as `x` / `y` respectively. No broadcasting in this drill — that's covered by cx30. Cross-check against `torch.autograd` so the elementwise chain-rule formula is verified, not just typed.

In [ ]:
def cx29_multiply_back(grad_out, out, x, y, argnum):
    # Elementwise chain rule: dL/dz = grad_out * (∂out/∂z).
    # For multiply: ∂(x*y)/∂x = y, and ∂(x*y)/∂y = x.
    # The 'OTHER operand IS the derivative' — that's why multiply_back is so clean.
    if argnum == 0:
        return grad_out * y
    if argnum == 1:
        return grad_out * x
    raise ValueError(f'multiply has args (x, y); argnum must be 0 or 1, got {argnum}')


<details><summary>Show solution — cx29</summary>

```python
def cx29_multiply_back(grad_out, out, x, y, argnum):
    # Elementwise chain rule: dL/dz = grad_out * (∂out/∂z).
    # For multiply: ∂(x*y)/∂x = y, and ∂(x*y)/∂y = x.
    # The 'OTHER operand IS the derivative' — that's why multiply_back is so clean.
    if argnum == 0:
        return grad_out * y
    if argnum == 1:
        return grad_out * x
    raise ValueError(f'multiply has args (x, y); argnum must be 0 or 1, got {argnum}')
```

The elementwise chain rule is the simplest case of backward: no transpose, no broadcasting (this drill skips broadcasting deliberately), just `grad_out * local-derivative`. For multiply the local derivative is the OTHER operand — that's why these two atoms collapse into a one-line formula per argnum. The same pattern generalizes to `div_back0 = grad_out / y`, `pow_back0 = grad_out * y * x**(y-1)`, etc.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx29'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx29',
        'subtopics': ["Backprop: multiply_back", "Backprop: Elementwise chain rule"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()